In [ ]:

import os
import shutil
import random
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── paths ──────────────────────────────────────────────────────────────────
xview_path   = Path('datasets/xView')
img_path   = xview_path / 'images' / 'train'   # xview .tif files
label_path   = xview_path / 'labels' / 'train'   # xview .txt files

tiles_path    = Path('datasets/construction_tiles') # output path of tiling step
tile_img_path  = tiles_path / 'images'
tile_labels_path  = tiles_path / 'labels'

# ── constants ─────────────────────────────────────────────────────────────
target_class  = 52          # construction site class in xView
tile_size     = 640         # img size for training
overlap       = 128         # overlap between tiles to avoid cutting them in half 
stride        = tile_size - overlap
norm_area  = 0.001       # skip labels whose normalised box area < 0.1%
val_split     = 0.15        # 15% of images held out for validation

print(f"Source images : {img_path}")
print(f"Source labels : {label_path}")
print(f"Tile output   : {tiles_path}")


In [ ]:
def load_labels(label_path: Path):
    labels = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                labels.append([float(p) for p in parts[:5]])
    return labels

In [ ]:


def tile_image(img_path: Path, label_path: Path, out_img_dir: Path, out_lbl_dir: Path):
    try:
        img = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"  [SKIP] Cannot open {img_path.name}: {e}")
        return 0

    W, H   = img.size
    labels = load_labels(label_path)

    boxes = []
    for cls, cx, cy, bw, bh in labels:
        if int(cls) != target_class:
            continue
        # absolute pixel coords (centre)
        abs_cx = cx * W
        abs_cy = cy * H
        abs_bw = bw * W
        abs_bh = bh * H
        x1 = abs_cx - abs_bw / 2
        y1 = abs_cy - abs_bh / 2
        x2 = abs_cx + abs_bw / 2
        y2 = abs_cy + abs_bh / 2
        boxes.append((x1, y1, x2, y2))

    if not boxes:
        return 0  # no construction sites in this image

    n_saved = 0
    stem    = img_path.stem

    for row, ty in enumerate(range(0, max(H - tile_size + 1, 1), stride)):
        for col, tx in enumerate(range(0, max(W - tile_size + 1, 1), stride)):
            tx2 = min(tx + tile_size, W)
            ty2 = min(ty + tile_size, H)
            tx1, ty1 = tx2 - tile_size, ty2 - tile_size  # handle edge

            tile_labels = []
            for (x1, y1, x2, y2) in boxes:
                # Clip box to tile
                cx1 = max(x1, tx1)
                cy1 = max(y1, ty1)
                cx2 = min(x2, tx2)
                cy2 = min(y2, ty2)

                if cx2 <= cx1 or cy2 <= cy1:
                    continue  # box doesn't intersect tile

                orig_area  = (x2 - x1) * (y2 - y1)
                clip_area  = (cx2 - cx1) * (cy2 - cy1)
                if orig_area > 0 and clip_area / orig_area < 0.25:
                    continue  # less than 25% visible — skip

                ncx = ((cx1 + cx2) / 2 - tx1) / tile_size
                ncy = ((cy1 + cy2) / 2 - ty1) / tile_size
                nw  = (cx2 - cx1) / tile_size
                nh  = (cy2 - cy1) / tile_size

                # Skip tiny boxes
                if nw * nh < norm_area:
                    continue

                tile_labels.append(f"0 {ncx:.6f} {ncy:.6f} {nw:.6f} {nh:.6f}")

            if not tile_labels:
                continue  # empty tile — skip

            tile_name = f"{stem}_r{row:03d}_c{col:03d}"
            tile_img  = img.crop((tx1, ty1, tx2, ty2))
            tile_img.save(out_img_dir / f"{tile_name}.png")
            (out_lbl_dir / f"{tile_name}.txt").write_text("\n".join(tile_labels))
            n_saved += 1

    return n_saved


# Tiling
if tiles_path.exists() and any(tile_img_path.glob('*.png')):
    n_tiles = len(list(tile_img_path.glob('*.png')))
    print(f"Tiles already exist, delete")
else:
    tile_img_path.mkdir(parents=True, exist_ok=True)
    tile_labels_path.mkdir(parents=True, exist_ok=True)

    tif_files = sorted(img_path.glob('*.tif'))

    total_tiles  = 0
    images_with_cs = 0
    for i, img_path in enumerate(tif_files):
        lbl_path = label_path / (img_path.stem + '.txt')
        n = tile_image(img_path, lbl_path, tile_img_path, tile_labels_path)
        if n > 0:
            images_with_cs += 1
            total_tiles    += n
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(tif_files)}")

    print(f"{images_with_cs} images had construction sites")


In [ ]:

# split by original imgs first (not tile) to avoid data leakage
all_tiles   = sorted(tile_img_path.glob('*.png'))
source_ids  = sorted({p.stem.rsplit('_r', 1)[0] for p in all_tiles})

random.shuffle(source_ids)
n_val      = max(1, int(len(source_ids) * val_split))
val_ids    = set(source_ids[:n_val])
train_ids  = set(source_ids[n_val:])

train_tiles = [p for p in all_tiles if p.stem.rsplit('_r', 1)[0] in train_ids]
val_tiles   = [p for p in all_tiles if p.stem.rsplit('_r', 1)[0] in val_ids]

# write split .txt files inside a folder 
split_dir = tiles_path / 'splits'
split_dir.mkdir(exist_ok=True)
(split_dir / 'train.txt').write_text('\n'.join(str(p.resolve()) for p in train_tiles))
(split_dir / 'val.txt').write_text('\n'.join(str(p.resolve()) for p in val_tiles))


In [ ]:
# %%
yaml_path = tiles_path / 'construction.yaml'
yaml_path.write_text(f"""#construction site detection
path: {tiles_path.resolve()}
train: splits/train.txt
val:   splits/val.txt

nc: 1
names:
  0: Construction Site
""")
print(yaml_path.read_text())



In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')  # smallest of the models, coz of gpu constraints
                            

results = model.train(
    data=str(yaml_path.resolve()),
    project='runs/construction',
    name='v1',
    device='cuda',

    epochs= 8,
    batch=16,
    imgsz=640,
    workers=0,

    # Augmentation 
    degrees=10.0,
    translate=0.1,   
    scale=0.3, 
    shear=5.0,
    perspective=0.0,  
    flipud=0.2,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.2,

    
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=3,
    patience=20,    # early stopping 
    cos_lr=True,

    # for vladiotion
    val=True,
    iou=0.5,
    exist_ok=True,
    verbose=True,
    conf=0.25, 
)


In [ ]:

import pandas as pd

results_csv = Path('runs/construction/v1/results.csv')
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Training Results', fontsize=14)

    plot_pairs = [
        ('train/box_loss',  'train/cls_loss', 'Loss'),
        ('metrics/precision(B)', 'metrics/recall(B)', 'Precision / Recall'),
        ('metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'mAP'),
    ]
    colors = ['steelblue', 'tomato']
    for ax_row, (col_a, col_b, title) in zip(axes, plot_pairs):
        for ax, col, color in zip(ax_row, [col_a, col_b], colors):
            if col in df.columns:
                ax.plot(df[col], color=color)
                ax.set_title(col.split('/')[-1])
                ax.set_xlabel('Epoch')
                ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    best_map = df['metrics/mAP50(B)'].max() if 'metrics/mAP50(B)' in df.columns else None
    print(f"Best mAP@50: {best_map:.3f}" if best_map else "No mAP data yet")
else:
    print("No results.csv found")



In [ ]:
%matplotlib inline
import cv2

best_model = model

# Pick a few validation tiles
sample_tiles = random.sample(val_tiles, min(8, len(val_tiles)))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for ax, tile_path in zip(axes, sample_tiles):
    res = best_model.predict(
        source=str(tile_path),
        imgsz=640,
        conf=0.25,
        iou=0.45,
        verbose=False,
    )[0]

    annotated = res.plot(labels=True, boxes=True, conf=True)
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    n_det = len(res.boxes)
    ax.imshow(annotated)
    ax.set_title(f"{tile_path.stem}\n{n_det} detection(s)", fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.suptitle('val predictions (conf >= 0.25)')
plt.show()
